[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/MarioSigal/TP_Rompecabezas/blob/main/TP_NIVEL_6.ipynb)

# 🧩 Trabajo Práctico: Resolución Automatizada de Rompecabezas
**Procesamiento de Imágenes (PDI) — Nivel 6 Integrador**

**Grupo:**

**Integrantes:**

---

## 🎯 Objetivo General
En este **Nivel 6 Integrador** deberán combinar e integrar todas las herramientas desarrolladas en los niveles anteriores para resolver rompecabezas afectados por múltiples degradaciones sintéticas simultáneas.

Cada rompecabezas en este nivel posee una base geométrica de:
- **Ruido Espacial Global (Nivel 1):** Gaussiano, Rayleigh, Uniforme o Sal y Pimienta.
- **Alteración Cromática por Pieza (Nivel 2):** Variaciones individuales de matiz o iluminación/valor.
- **Ruido Periódico en Frecuencia (Nivel 3):** Tramas sinusoidales armónicas (muaré) filtrables mediante Fourier 2D.
- **Encastres Curvos (Nivel 4)** y puede contener de forma combinada y aleatoria:
- **Rotación con Rayas Espectrales (Nivel 5):** Inclinación aleatoria de piezas con modulación armónica para deskewing en Fourier 2D.

---

### Información:
1. **Exclusión Mutua Fourier (Nivel 3 y Nivel 5):**
   * El **Ruido Periódico (Nivel 3)** y la **Rotación con Rayas (Nivel 5)** **NUNCA se combinan en un mismo rompecabezas**. Esto garantiza que en el espectro 2D de Fourier los picos armónicos de las rayas de orientación no se confundan con las frecuencias del ruido periódico.
2. **Exclusión de Ruido Impulsivo con Fourier:**
   * Si la pieza posee ruido periódico (Nivel 3), nunca tendrá ruido Sal & Pimienta (para evitar la dispersión de alta frecuencia en el espectro).


---
## ⚙️ Configuración del Entorno de Ejecución

Esta celda configura automáticamente las rutas necesarias tanto si se ejecuta en **Google Colab** como en un entorno local de Jupyter.


In [ ]:
# Configuración de entorno para Google Colab y ejecución local
import os, sys
from pathlib import Path

if 'google.colab' in str(get_ipython()):
    print('--> Entorno detectado: Google Colab')
    if not os.path.exists('repo_tp') and not os.path.exists('core'):
        !git clone https://github.com/MarioSigal/TP_Rompecabezas.git repo_tp
        %cd repo_tp
    else:
        if os.path.exists('repo_tp'):
            %cd repo_tp
        !git pull origin main
    sys.path.insert(0, os.getcwd())
else:
    print('--> Entorno detectado: Local')
    raiz = Path.cwd()
    if (raiz / 'TP_FINAL_ALUMNOS' / 'core').exists():
        sys.path.insert(0, str(raiz / 'TP_FINAL_ALUMNOS'))
    elif (raiz / 'core').exists():
        sys.path.insert(0, str(raiz))
    elif (raiz / 'TP_Rompecabezas' / 'core').exists():
        sys.path.insert(0, str(raiz / 'TP_Rompecabezas'))

import numpy as np
import cv2
import matplotlib.pyplot as plt

# Importar funciones del núcleo del TP
from core import (
    cargar_imagen,
    guardar_imagen,
    preparar_imagen_base,
    crear_rompecabezas_nivel,
    reconstruir_rompecabezas,
    reconstruir_desde_afinidades,
    construir_matrices_afinidad,
    compatibilidad_baseline,
    segmentar_borde_en_4,
    binarize_piece,
    extract_external_contour,
    estimar_orientacion_fourier,
    enderezar_pieza,
    generar_reporte_completo,
    imprimir_reporte,
)

# Tabla de encastres discretos (opcional)
try:
    from core import TABLA_BORDES_DISCRETOS
except ImportError:
    try:
        from core.geometria_jigsaw import TABLA_BORDES_DISCRETOS
    except ImportError:
        TABLA_BORDES_DISCRETOS = None

from utils import (
    mostrar_piezas_desordenadas,
    mostrar_comparacion_imagen,
    mostrar_espectro_fourier,
    mostrar_reconstruccion,
    crear_animacion,
)

print('✅ ¡Módulos del TP cargados con éxito!')


---
## 🖼️ Configuración y Carga de Imágenes para el Desafío


In [ ]:
# Selección de imágenes de prueba para el Nivel 6
EXTENSIONES_VALIDAS = {".png", ".jpg", ".jpeg", ".bmp"}

# Buscar carpeta de imágenes disponible (dataset desafío o base)
CARPETA_DATASET = Path("imagenes/dataset_desafio")
if not CARPETA_DATASET.exists() or not any(p for p in CARPETA_DATASET.iterdir() if p.suffix.lower() in EXTENSIONES_VALIDAS):
    CARPETA_DATASET = Path("imagenes/base")
if not CARPETA_DATASET.exists():
    CARPETA_DATASET = Path("TP_FINAL_ALUMNOS/imagenes/base")

rutas_imagenes = sorted(
    p for p in CARPETA_DATASET.iterdir()
    if p.suffix.lower() in EXTENSIONES_VALIDAS
)

print(f"Total de imágenes encontradas en {CARPETA_DATASET}: {len(rutas_imagenes)}")

# Mapeo de variantes cromáticas y semillas de prueba para evaluación
IMAGENES_EVALUACION = [p.name for p in rutas_imagenes[:2]] if rutas_imagenes else ["paisaje.png"]

VARIANTE_CROMATICA_POR_IMAGE = {
    nombre: ("matiz" if i % 2 == 0 else "valor")
    for i, nombre in enumerate(IMAGENES_EVALUACION)
}

SEMILLA_POR_IMAGEN = {
    nombre: 100 + i * 111
    for i, nombre in enumerate(IMAGENES_EVALUACION)
}

print("Imágenes seleccionadas para evaluación:")
for img_name in IMAGENES_EVALUACION:
    print(f"  - {img_name}: Semilla={SEMILLA_POR_IMAGEN[img_name]}, Variante Cromática={VARIANTE_CROMATICA_POR_IMAGE[img_name]}")


### 🧩 Creación de Puzzles Integradores Nivel 6
Cada rompecabezas posee encastres discretos y una combinación aleatoria de degradaciones cumpliendo las reglas de exclusión.


In [ ]:
# Generación de puzzles Nivel 6
puzzles = []

for nombre_img in IMAGENES_EVALUACION:
    ruta_img = next(p for p in rutas_imagenes if p.name == nombre_img)
    imagen = preparar_imagen_base(ruta_img, tamaño_objetivo=(640, 640))
    variante_cromatica = VARIANTE_CROMATICA_POR_IMAGE[nombre_img]
    semilla = SEMILLA_POR_IMAGEN[nombre_img]

    puzzle = crear_rompecabezas_nivel(
        imagen,
        nivel=6,
        filas=4,
        columnas=4,
        semilla=semilla,
        variante_cromatica=variante_cromatica,
        discrete=True,
    )
    puzzles.append(puzzle)
    meta = puzzle.metadatos
    print(f"🧩 Puzzle [{nombre_img}] | Semilla: {semilla} | Problemas activos: {meta.get('problemas_activos')}")
    print(f"   Ruido Global: {meta.get('tiene_ruido_global')} | Color: {meta.get('tiene_color')} | Fourier: {meta.get('tiene_fourier')} | Rotación: {meta.get('tiene_rotacion')}")

# Visualizar piezas desordenadas del primer rompecabezas
mostrar_piezas_desordenadas(puzzles[0], max_piezas=12, titulo=f'Piezas Nivel 6 ({IMAGENES_EVALUACION[0]})')


---
## 🛠️ Pipeline de Diagnóstico, Limpieza y Métrica Combinada

En este bloque se definen las etapas secuenciales del pipeline de restauración y ensamblado:
1. **Deskewing (Nivel 5):** Si la pieza tiene modulación de rayas y rotación, se estima el ángulo con Fourier 2D y se endereza a 0°.
2. **Filtro Notch (Nivel 3):** Si la pieza tiene ruido periódico sinusoidal, se detectan y anulan los picos de frecuencia fuera de DC.
3. **Filtrado Espacial Adaptativo (Nivel 1):** Mediana para sal y pimienta; Gaussiano para ruido continuo.
4. **Métrica Combinada Forma + MSE de Color (Nivel 4):**
   $$\text{costo}(A, B) = \text{costo\_forma}(A, B) + \lambda \cdot \text{MSE\_color}(A, B)$$


In [ ]:
# 1. Detección y Filtrado Espacial de Ruido (Nivel 1)
def detectar_tipo_ruido(piezas: list) -> dict:
    """
    Diagnostica si el conjunto de piezas presenta ruido impulsivo (sal/pimienta)
    o ruido continuo (gaussiano/rayleigh/uniforme).
    """
    cuenta_sal = 0
    cuenta_pimienta = 0
    varianzas = []

    for p in piezas[:min(8, len(piezas))]:
        mask = (p.max(axis=2) > 0.01) if p.ndim == 3 else (p > 0.01)
        vals = p[mask]
        if len(vals) == 0:
            continue
        cuenta_sal += np.sum(vals > 0.98)
        cuenta_pimienta += np.sum(vals < 0.02)
        varianzas.append(float(np.var(vals)))

    total_px = max(1, sum(np.sum((p.max(axis=2) > 0.01) if p.ndim == 3 else (p > 0.01)) for p in piezas[:min(8, len(piezas))]))
    frac_sal = cuenta_sal / total_px
    frac_pimienta = cuenta_pimienta / total_px
    var_media = float(np.mean(varianzas)) if varianzas else 0.0

    tiene_sal = (frac_sal > 0.005)
    tiene_pimienta = (frac_pimienta > 0.005)
    tiene_continuo = (var_media > 0.015)

    return {
        "tiene_sal": tiene_sal,
        "tiene_pimienta": tiene_pimienta,
        "tiene_continuo": tiene_continuo,
    }


def filtrar_ruido_espacial(pieza: np.ndarray, diag: dict) -> np.ndarray:
    """
    Limpia el ruido espacial adaptando el filtro según el diagnóstico.
    Conserva rigurosamente el fondo negro fuera de la silueta.
    """
    mask = (pieza.max(axis=2) > 0.01) if pieza.ndim == 3 else (pieza > 0.01)
    p_u8 = np.round(np.clip(pieza, 0.0, 1.0) * 255.0).astype(np.uint8)
    res = p_u8.copy()

    if diag.get("tiene_sal") or diag.get("tiene_pimienta"):
        res = cv2.medianBlur(res, 3)

    if diag.get("tiene_continuo"):
        res = cv2.GaussianBlur(res, (3, 3), 0.8)

    res_f = res.astype(np.float64) / 255.0
    res_f[~mask] = 0.0
    return res_f


def filtrar_fourier_notch(pieza: np.ndarray, radio_dc: int = 12, radio_muesca: int = 4) -> np.ndarray:
    """
    Aplica filtro Notch automático en Fourier 2D si se detectan picos armónicos de ruido periódico.
    """
    gray = cv2.cvtColor((np.clip(pieza, 0.0, 1.0) * 255.0).astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)
    F = np.fft.fftshift(np.fft.fft2(gray))
    mag = np.abs(F)

    cy, cx = mag.shape[0] // 2, mag.shape[1] // 2
    mag_sin_dc = mag.copy()
    mag_sin_dc[cy - radio_dc : cy + radio_dc + 1, cx - radio_dc : cx + radio_dc + 1] = 0.0

    umbral = np.mean(mag_sin_dc) + 4.5 * np.std(mag_sin_dc)
    picos_y, picos_x = np.where(mag_sin_dc > umbral)

    if len(picos_y) == 0:
        return pieza

    H = np.ones(gray.shape, dtype=np.float32)
    for py, px in zip(picos_y, picos_x):
        cv2.circle(H, (px, py), radio_muesca, 0.0, -1)

    canales_filtrados = []
    mask = (pieza.max(axis=2) > 0.01) if pieza.ndim == 3 else (pieza > 0.01)

    for ch in range(3):
        F_ch = np.fft.fftshift(np.fft.fft2(pieza[:, :, ch]))
        F_filt = F_ch * H
        inv = np.real(np.fft.ifft2(np.fft.ifftshift(F_filt)))
        canales_filtrados.append(np.clip(inv, 0.0, 1.0))

    res = np.stack(canales_filtrados, axis=2)
    res[~mask] = 0.0
    return res


def enderezar_si_tiene_rotacion(piezas: list, puzzle) -> list:
    """
    Si el rompecabezas posee rotación (Nivel 5), endereza cada pieza buscando
    el pico armónico de las rayas en Fourier 2D.
    """
    if not puzzle.metadatos.get("tiene_rotacion", False):
        return piezas

    piezas_rect = []
    for p in piezas:
        ang = estimar_orientacion_fourier(p)
        p_rect, _ = enderezar_pieza(p, angulo_grados=ang, padding=0)
        piezas_rect.append(p_rect)
    return piezas_rect


In [ ]:
# Métrica de compatibilidad combinada: Silueta Geométrica + MSE de Color en el Borde
def mi_compatibilidad_integrador(pieza_a: np.ndarray, pieza_b: np.ndarray, relacion: str, peso_color: float = 10.0) -> float:
    """
    Evalúa la compatibilidad entre dos piezas combinando morfología y color:
    1. Segmenta ambas piezas con 'segmentar_borde_en_4' para extraer los lados enfrentados
       con su señal 1D de forma y su perfil de color RGB ('color_profile').
    2. Evalúa complementariedad geométrica (macho con hembra).
    3. Si encastran, desempata entre piezas con la misma forma discreta calculando
       el MSE de color de la costura mediante 'compatibilidad_baseline(lado_a, lado_b)'.
    """
    BORDES_OPUESTOS = {
        "horizontal": ("ESTE", "OESTE"),
        "vertical": ("SUR", "NORTE"),
    }
    if relacion not in BORDES_OPUESTOS:
        raise ValueError(f"Relación inválida: '{relacion}'. Se espera 'horizontal' o 'vertical'.")

    nom_a, nom_b = BORDES_OPUESTOS[relacion]
    lados_a = segmentar_borde_en_4(pieza_a)
    lados_b = segmentar_borde_en_4(pieza_b)
    lado_a = lados_a[nom_a]
    lado_b = lados_b[nom_b]
    tipo_a = lado_a["type"]
    tipo_b = lado_b["type"]

    # 1. Chequeo morfológico de encastre: dos bordes interiores enfrentados no pueden ser PLANO
    if tipo_a == "PLANO" or tipo_b == "PLANO":
        return 1e5

    is_sal_a = tipo_a in ("SALIENTE", "MACHO", "PESTAÑA")
    is_ent_a = tipo_a in ("ENTRANTE", "HEMBRA", "HENDIDURA", "MUESCA")
    is_sal_b = tipo_b in ("SALIENTE", "MACHO", "PESTAÑA")
    is_ent_b = tipo_b in ("ENTRANTE", "HEMBRA", "HENDIDURA", "MUESCA")

    if not ((is_sal_a and is_ent_b) or (is_ent_a and is_sal_b)):
        return 1e5

    prof_a = lado_a["profile"]
    prof_b = lado_b["profile"]
    norm_a = lado_a.get("norm", float(np.linalg.norm(prof_a)))
    norm_b = lado_b.get("norm", float(np.linalg.norm(prof_b)))

    if norm_a < 1e-4 or norm_b < 1e-4:
        return 1e5

    dot = float(np.dot(prof_a, -prof_b))
    rho = dot / (norm_a * norm_b)
    if rho <= 0.0:
        return 1e5

    amp_ratio = min(norm_a, norm_b) / max(norm_a, norm_b)
    corr = rho * amp_ratio
    if corr <= 1e-3:
        return 1e5

    costo_forma = (1.0 - corr) * 10.0

    # 2. Desempate cromático con compatibilidad_baseline usando los perfiles de color del borde
    costo_color = compatibilidad_baseline(lado_a["color_profile"], lado_b["color_profile"])

    return costo_forma + peso_color * costo_color


---
## 🚀 Corrida Final y Evaluación por Puzzle


In [ ]:
def get_reporte_por_puzzle(
    puzzle,
    funcion_compatibilidad=mi_compatibilidad_integrador,
):
    """
    Ejecuta el pipeline completo de resolución sobre un rompecabezas de Nivel 6:
    1. Deskewing de orientación si posee Nivel 5.
    2. Supresión de ruido periódico si posee Nivel 3.
    3. Filtrado espacial adaptativo si posee Nivel 1.
    4. Cálculo de matrices de afinidad con la métrica combinada forma+color.
    5. Reconstrucción global y generación del reporte de métricas.
    """
    piezas = puzzle.piezas

    # Paso 1: Deskewing (solo si tiene rotación de Nivel 5)
    piezas = enderezar_si_tiene_rotacion(piezas, puzzle)

    # Paso 2: Filtrado en frecuencia (solo si tiene ruido periódico de Nivel 3)
    if puzzle.metadatos.get("tiene_fourier", False):
        piezas = [filtrar_fourier_notch(p) for p in piezas]

    # Paso 3: Filtrado espacial adaptativo (Nivel 1)
    if puzzle.metadatos.get("tiene_ruido_global", False):
        diag = detectar_tipo_ruido(piezas)
        piezas = [filtrar_ruido_espacial(p, diag) for p in piezas]

    # Paso 4: Construcción de matrices de afinidad con la métrica combinada
    matrices_afinidad = construir_matrices_afinidad(piezas, funcion_compatibilidad)

    # Paso 5: Reconstrucción
    grilla, rec = reconstruir_desde_afinidades(
        matrices_afinidad,
        puzzle.cantidad_filas,
        puzzle.cantidad_columnas,
        devolver_reconstructor=True,
    )

    reporte = generar_reporte_completo(puzzle, matrices_afinidad=matrices_afinidad, grilla_propuesta=grilla)
    return {
        "reporte": reporte,
        "grilla": grilla,
        "piezas_procesadas": piezas,
        "reconstructor": rec,
    }


In [ ]:
resultados = []

for idx, puzzle in enumerate(puzzles):
    print(f"\n🧩 Resolviendo Rompecabezas {idx + 1}/{len(puzzles)}: {IMAGENES_EVALUACION[idx]}...")
    res = get_reporte_por_puzzle(puzzle, funcion_compatibilidad=mi_compatibilidad_integrador)
    resultados.append(res)
    imprimir_reporte(res["reporte"], titulo=f'Reporte Puzzle {idx + 1} ({IMAGENES_EVALUACION[idx]})')

# Mostrar la reconstrucción del primer puzzle
if resultados:
    mostrar_reconstruccion(
        puzzles[0],
        resultados[0]["grilla"],
        piezas=resultados[0]["piezas_procesadas"],
        titulo=f"Reconstrucción Final: {IMAGENES_EVALUACION[0]}"
    )


In [ ]:
def calcular_reporte_promedio(reportes: list) -> dict:
    """
    Promedia las métricas numéricas principales (top1_promedio, precision_vecindad, precision_directa)
    sobre todos los rompecabezas evaluados.
    """
    if not reportes:
        return {}

    suma = {}
    conteo = {}
    for reporte in reportes:
        for clave, valor in reporte.items():
            if not isinstance(valor, (int, float)):
                continue
            suma[clave] = suma.get(clave, 0.0) + valor
            conteo[clave] = conteo.get(clave, 0) + 1

    return {clave: round(suma[clave] / conteo[clave], 4) for clave in suma}


---
## 🏆 Puntaje Total de la Reconstrucción en Nivel 6


In [ ]:
promedios = calcular_reporte_promedio([r["reporte"] for r in resultados])
print("=" * 50)
print("📊 RESUMEN PROMEDIO NIVEL 6 INTEGRADOR")
print("=" * 50)
print(f"  - Top-1 Vecino Más Cercano : {promedios.get('top1_promedio', 0.0) * 100:.2f}%")
print(f"  - Precisión de Vecindad     : {promedios.get('precision_vecindad', 0.0) * 100:.2f}%")
print(f"  - Precisión Directa         : {promedios.get('precision_directa', 0.0) * 100:.2f}%")
print("=" * 50)
